# Multiple simulation demo

This notebook runs many randomized multi-year simulations using the same sampled start dates for two strategies:

- Annual rebalancing
- Buy and hold

Both strategies use the same equal-weight starting allocation and dividend reinvestment behavior.

It then reports:

- Per-strategy overall results
- Distribution of total returns by strategy
- Distribution of Sortino ratios by strategy
- Asset and portfolio value charts for best and worst annual-rebalance runs

In [ ]:
from datetime import date
from importlib import reload

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from investing import history
from investing import history as history_mod
from investing import reporting
from investing import simulation
from investing.portfolio import AssetAllocation, HoldingTarget

reload(history)
reload(history_mod)
reload(reporting)
reload(simulation)

sns.set_theme(style="whitegrid")

In [ ]:
market_history = history_mod.load_market_history(
    "../data/5-way-prices.xlsx",
    "../data/5-way-dividends.xlsx",
)

sorted_tickers = sorted(market_history.securities.keys())
allocation = AssetAllocation([HoldingTarget(ticker, 1) for ticker in sorted_tickers])
rebalance_strategy = simulation.AnnualRebalance(allocation, max_deviation=0.05)
buy_and_hold_strategy = simulation.BuyAndHold(allocation)
strategies = [rebalance_strategy, buy_and_hold_strategy]

sorted_tickers

In [ ]:
YEARS = 4
START_FUNDS = 100_000
NUM_SIMULATIONS = 500
SEED = 42
PLAN_TARGET_RETURN = 0.04

multi = simulation.simulate_many(
    strategy=strategies,
    history=market_history,
    years=YEARS,
    start_funds=START_FUNDS,
    num_simulations=NUM_SIMULATIONS,
    seed=SEED,
    show_progress=True,
)

strategy_labels = list(multi.by_strategy.keys())
annual_label = next(label for label in strategy_labels if label.startswith("AnnualRebalance"))
buy_and_hold_label = next(label for label in strategy_labels if label.startswith("BuyAndHold"))

strategy_labels

In [ ]:
run_results_by_strategy: dict[str, pl.DataFrame] = {}
all_run_rows: list[dict[str, object]] = []

for strategy_label, strategy_result in multi.by_strategy.items():
    run_rows: list[dict[str, object]] = []
    for idx, (run, run_metrics) in enumerate(
        zip(strategy_result.simulations, strategy_result.run_metrics)
    ):
        start_portfolio = run.portfolios[0]
        end_portfolio = run.portfolios[-1]

        start_value = float(
            start_portfolio.total_value(start_portfolio.as_of_date, market_history)
        )
        end_value = float(end_portfolio.total_value(end_portfolio.as_of_date, market_history))
        total_return = end_value / start_value - 1.0

        row = {
            "strategy": strategy_label,
            "run": idx,
            "start_date": start_portfolio.as_of_date,
            "end_date": end_portfolio.as_of_date,
            "start_value": start_value,
            "end_value": end_value,
            "total_return": total_return,
            "sortino_ratio": run_metrics.sortino_ratio,
            "cagr": run_metrics.cagr,
            "max_drawdown": run_metrics.max_drawdown,
            "std_dev_returns": run_metrics.std_dev_returns,
        }
        run_rows.append(row)
        all_run_rows.append(row)

    run_results_by_strategy[strategy_label] = pl.DataFrame(run_rows)

run_results = pl.DataFrame(all_run_rows)
run_results.head()

In [ ]:
overall_rows: list[dict[str, object]] = []
for strategy_label, strategy_result in multi.by_strategy.items():
    strategy_runs = run_results_by_strategy[strategy_label]
    overall_rows.append(
        {
            "strategy": strategy_label,
            "num_simulations": len(strategy_result.simulations),
            "years": YEARS,
            "start_funds": START_FUNDS,
            "plan_target_return": PLAN_TARGET_RETURN,
            "avg_total_return": float(strategy_runs["total_return"].mean()),
            "median_total_return": float(strategy_runs["total_return"].median()),
            "min_total_return": float(strategy_runs["total_return"].min()),
            "max_total_return": float(strategy_runs["total_return"].max()),
            "avg_sortino_ratio": None
            if (
                avg_sortino_ratio := strategy_runs["sortino_ratio"].drop_nulls().mean()
            )
            is None
            else float(avg_sortino_ratio),
            "median_sortino_ratio": None
            if (
                median_sortino_ratio := strategy_runs["sortino_ratio"].drop_nulls().median()
            )
            is None
            else float(median_sortino_ratio),
            "aggregate_cagr": strategy_result.metrics.cagr,
            "aggregate_max_drawdown": strategy_result.metrics.max_drawdown,
            "aggregate_std_dev_returns": strategy_result.metrics.std_dev_returns,
            "aggregate_sortino_ratio": strategy_result.metrics.sortino_ratio,
            "aggregate_success_probability": strategy_result.metrics.success_probability,
            "terminal_wealth_p10": strategy_result.metrics.terminal_wealth_p10,
            "terminal_wealth_p50": strategy_result.metrics.terminal_wealth_p50,
            "terminal_wealth_p90": strategy_result.metrics.terminal_wealth_p90,
        }
    )

overall_results = pl.DataFrame(overall_rows)
overall_results

In [ ]:
with pl.Config(tbl_rows=overall_results.height):
    display(overall_results.sort("strategy"))

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    run_results.to_pandas(),
    x="total_return",
    hue="strategy",
    bins=30,
    stat="density",
    common_norm=False,
    kde=True,
    alpha=0.35,
)
plt.title(f"Distribution of {YEARS}-Year Total Returns by Strategy")
plt.xlabel("Total return")
plt.ylabel("Density")
plt.show()

In [ ]:
sortino_values = run_results.select(["strategy", "sortino_ratio"]).drop_nulls().to_pandas()

if sortino_values.empty:
    print("No non-null Sortino ratios available for this simulation window.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(
        sortino_values,
        x="sortino_ratio",
        hue="strategy",
        bins=30,
        stat="density",
        common_norm=False,
        kde=True,
        alpha=0.35,
    )
    plt.title("Distribution of Sortino Ratios by Strategy")
    plt.xlabel("Sortino ratio")
    plt.ylabel("Density")
    plt.show()

In [ ]:
annual_runs = run_results_by_strategy[annual_label]
annual_multi = multi.by_strategy[annual_label]

best_run_id = int(annual_runs.sort("end_value", descending=True)["run"][0])
worst_run_id = int(annual_runs.sort("end_value", descending=False)["run"][0])

best_values = reporting.value_history(
    annual_multi.simulations[best_run_id].portfolios,
    market_history,
    "monthly",
)
worst_values = reporting.value_history(
    annual_multi.simulations[worst_run_id].portfolios,
    market_history,
    "monthly",
)

plt.figure(figsize=(12, 5))
sns.lineplot(
    best_values.filter(pl.col("ticker") != "_TOTAL"),
    x="date",
    y="valuation",
    hue="ticker",
)
plt.title(f"Best annual-rebalance run asset values (run {best_run_id})")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(
    best_values.filter(pl.col("ticker") == "_TOTAL"),
    x="date",
    y="valuation",
    color="black",
)
plt.title(f"Best annual-rebalance run portfolio value (run {best_run_id})")
plt.xlabel("Date")
plt.ylabel("Total value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(
    worst_values.filter(pl.col("ticker") != "_TOTAL"),
    x="date",
    y="valuation",
    hue="ticker",
)
plt.title(f"Worst annual-rebalance run asset values (run {worst_run_id})")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(
    worst_values.filter(pl.col("ticker") == "_TOTAL"),
    x="date",
    y="valuation",
    color="black",
)
plt.title(f"Worst annual-rebalance run portfolio value (run {worst_run_id})")
plt.xlabel("Date")
plt.ylabel("Total value")
plt.tight_layout()
plt.show()